# 실전 데이터 분석 69: 오프라인 소매 상점의 기상 기후 및 세일 행사 여부 대비 유입 방문객 수와 하루 매출액 분석

> **📥 [실습 주피터 노트북(.ipynb) 다운로드](practice.ipynb)**

## 📌 강의 개요 (30분 완성)

![코믹 일러스트](img/intro_comic.png)

오프라인 대형 리테일 샵의 일일 방문자 유입 및 매출 기록 데이터셋입니다. 당일 기상 조건(Weather)과 파격 세일 행사 여부(DiscountEvent)가 방문객 볼륨 및 최종 일 매출에 미치는 효과를 시각화합니다.

**학습 목표:**
* **날씨 조건부 평균 대치 (`groupby.transform`):** 비가 오거나 흐린 날의 기저 매출이 맑은 날과 상이하므로 날씨별 평균 매출로 누락값을 정밀 대치합니다.
* **방문객 유입 및 세일 시너지 산점도 (`barplot`, `scatterplot`):** 날씨 조건별 일일 평균 방문자 막대 및 유입량 대비 일 매출액의 세일 조건별 상관 지도를 도출합니다.

---

## Step 1: 데이터 구조 살펴보기 (Data Overview)

![Step 1 데이터 구조 개념도](img/step1_dataset.svg)

`csv_data` 폴더에 준비해 둔 `store_traffic.csv` 파일을 판다스로 불러옵니다.


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# 그래프 설정 (한글 폰트 및 마이너스 기호 깨짐 방지)
import koreanize_matplotlib
plt.rcParams['axes.unicode_minus'] = False
sns.set_theme(style="whitegrid")

# 로컬 CSV 파일 불러오기
df = pd.read_csv('./store_traffic.csv')

# 데이터 구조 및 첫 5행 확인
df = pd.read_csv('./store_traffic.csv')
print(df.info())
print(df.head())


<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Date           1000 non-null   str    
 1   Weather        1000 non-null   str    
 2   VisitorCount   1000 non-null   int64  
 3   DailySales     986 non-null    float64
 4   DiscountEvent  1000 non-null   str    
dtypes: float64(1), int64(1), str(3)
memory usage: 39.2 KB
Date Weather  VisitorCount  DailySales DiscountEvent
0  2023-01-01   Sunny           561    11666.07            No
1  2023-01-02   Rainy           486     6430.62           Yes
2  2023-01-03   Sunny           580    16099.91            No
3  2023-01-04  Cloudy           449    13359.85            No
4  2023-01-05  Cloudy           408     7150.75            No
> ```

### 💡 코드 딥다이브 (Code Deep Dive)
**주요 분석 대상 컬럼:**
* `Date`: 일일 매출이 관측 기록된 날짜 정보 (YYYY-MM-DD)
* `Weather`: 해당 날짜의 주요 기상 상태 (Sunny: 맑음, Cloudy: 흐림, Rainy: 비, Snowy: 눈)
* `VisitorCount`: 당일 매장을 최종 유입 방문한 총 고객 수
* `DailySales`: 당일 포스기로 결제된 총 매장 매출액 (USD) (결측치 존재)
* `DiscountEvent`: 당일 파격 할인 프로모션 세일 진행 여부 (Yes/No)

---

## Step 2: 전처리와 결측치 정제 (Preprocess)

![Step 2 데이터 정제 개념도](img/step2_missing_values.svg)

현실의 데이터는 항상 누락이 있거나 유효성 정제가 필요합니다. 데이터 전처리 단계에서 결측 상태를 확인하고 올바르게 보정합니다.


In [ ]:
# 1. 컬럼별 결측치 개수 확인
print("--- 정제 전 결측치 확인 ---")
print(df.isnull().sum())

# 2. 결측치 전처리 및 대치 수행
df['DailySales'] = df.groupby('Weather')['DailySales'].transform(lambda x: x.fillna(x.mean()))
print(df.isnull().sum())


--- 정제 전 결측치 확인 ---
Date              0
Weather           0
VisitorCount      0
DailySales       14
DiscountEvent     0

--- 정제 후 결측치 확인 ---
Date             0
Weather          0
VisitorCount     0
DailySales       0
DiscountEvent    0
> ```

### 💡 분석가의 통찰 (Analyst's Insight)
* **기상 가중치 대입의 비즈니스적 당위성:** 비가 억수같이 내리거나 폭설이 내린 날은 기본 오프라인 방문객과 매출이 급감합니다. 이런 날 발생한 매출 결측을 맑은 주말 매출까지 섞인 1년 전체 평균으로 일괄 채우면 기상 오염 분석이 무너집니다. 날씨별 평균 매출을 각각 대치하는 이유입니다.

---

## Step 3: 단변수 분포 분석 (Univariate EDA)

![Step 3 시각화 개념도](img/step3_visualization.svg)

가장 먼저 핵심 변수가 전체 데이터에서 어떤 빈도와 분포를 가졌는지 단일 변수 시각화를 통해 파악해 봅니다.


In [ ]:
plt.figure(figsize=(8, 5))

# 단변수 분석 그래프 생성
sns.barplot(data=df, x='Weather', y='VisitorCount', palette='coolwarm', errorbar=None)
plt.title('기상 조건별 일일 평균 매장 방문객 수', fontsize=14, fontweight='bold')
plt.show()


> **💻 [실행 결과 시각화]**
> ![실행 결과 시각화](img/exec_step_3.svg)

### 💡 시각화 차트 읽는 법 & 인사이트
* **날씨가 오프라인 방문에 미치는 물리적 억제 효과:** 맑은 날(Sunny)과 흐린 날(Cloudy)의 막대 높이가 비가 오거나(Rainy) 눈이 내리는(Snowy) 궂은 날씨의 일일 방문객 유입 수준 대비 약 2~3배 높게 치솟아 있습니다. 이는 오프라인 매장의 운영 및 재고 관리가 당일 기상 예보와 매우 기민하게 연동되어 조율되어야 함을 증명합니다.

---

## Step 4: 다변수 상관관계 및 이상치 분석 (Multivariate EDA)

![Step 4 상관관계 분석 개념도](img/step4_multivariate.svg)

두 개 이상의 변수를 동시에 결합하여, 조건에 따른 수치 차이나 독립 변수와 종속 변수 간의 통계적 경향을 분석합니다.


In [ ]:
plt.figure(figsize=(9, 6))

# 다변수 분석 그래프 생성
sns.scatterplot(data=df, x='VisitorCount', y='DailySales', hue='DiscountEvent', palette='Set1', alpha=0.8)
plt.title('방문객 수와 일일 총 매출액의 세일 연계 상관성', fontsize=14, fontweight='bold')
plt.show()


> **💻 [실행 결과 시각화]**
> ![실행 결과 시각화](img/exec_step_4.svg)

### 💡 코드 딥다이브 & 비즈니스 통찰 (Analyst's Insight)
* **방문 유입과 매출의 선형 관계 및 세일 행사 시너지:** 방문객 규모(X축)가 증가함에 따라 일 매출(Y축)도 정직하고 가파르게 우상향합니다. 특히 할인 행사 진행일(DiscountEvent=Yes, 빨간 점)은 상대적으로 적은 유입 유모 조건하에서도 가파른 객단가 및 충동구매 유도로 매출 효율을 훨씬 위쪽 높이로 크게 뽑아내는 경향을 가시화합니다.

---

## Step 5: 통계적 직관과 해석 (Statistical Logic)

> 💡 **[비즈니스 분석에서 인과관계(Causality) vs 상관관계(Correlation)]**
> 오프라인 매장 매출 분석 시 가장 자주 저지르는 함정이 **인과관계와 상관관계의 혼동**입니다.
> * 예컨대 '당일 매장 방문객(X)과 매출(Y)이 높은 상관성을 갖는다'는 사실이 '방문객 수 증가가 무조건 매출액 상승의 단독 원인이다'를 확정짓지는 못합니다. 
> * 사실은 '할인 행사 여부(Discount)'라는 제3의 변수가 방문객 유입도 늘리고 동시에 구매율도 촉진하여 두 변수를 춤추게 만드는 **공통 요인(Confounder)**일 수 있기 때문입니다.
> * 따라서 데이터 사이언티스트는 이 관계를 단순 결합해 단정하기보다 세일 조건별로 통제 층화 분석(Stratified Analysis)을 거치고, 최종적으로 외생적인 프로모션을 독립 실행해 인과 효과의 순수한 크기를 발라내야 합니다.

---

## 🎯 30분 강의 마무리 및 심화 과제

오늘 우리는 실전 데이터셋을 분석하여 판다스로 데이터를 가공 및 정제하고, 시각화를 활용하여 핵심 변수 간의 통계적 유의성을 검증했습니다. 데이터 속에서 숨겨진 패턴을 올바른 시각으로 탐색하는 능력이 데이터 사이언티스트의 가장 강력한 무기입니다.

### 📝 심화 과제 (Advanced Challenge)
1. **기상 조건(Weather) 및 세일(DiscountEvent) 교차 집계:** `pd.pivot_table`을 이용해 날씨별, 세일 여부별 평균 매출액 테이블을 한눈에 볼 수 있게 교차 작성해 보세요.
2. **방문자당 평균 구매액(객단가) 요약:** 맑은 날(Sunny) 대비 비 오는 날(Rainy)의 고객 1인당 평균 결제 가치(`DailySales / VisitorCount`) 격차를 요약 대조해 보세요.
